# 06 — Model Improvement

## 1. Objective

The goal of this notebook is to improve the current best model, XGBoost, based on the error analysis findings.

We will test:
- class imbalance handling;
- probability threshold adjustment;
- hyperparameter tuning.

Model improvements will be evaluated mainly using Macro F1 and Class 1 Recall.

## 2. Evaluate the Current XGBoost Model with Cross-Validation

Before testing improvements, we evaluate the current XGBoost model using 5-fold stratified cross-validation.

This provides a robust reference for comparing all improvement experiments.

The primary evaluation metric is Macro F1.

In [7]:
import pandas as pd

In [3]:
import sys
sys.path.append("..")

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from xgboost import XGBClassifier

from src.data_split import load_and_split_data
from src.preprocessing import build_tree_preprocessor

X_train, X_valid, y_train, y_valid = load_and_split_data()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

baseline_xgb = Pipeline(
    steps=[
        ("preprocessor", build_tree_preprocessor()),
        ("model", XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            random_state=42,
            eval_metric="logloss"
        ))
    ]
)

scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro",
    "class1_recall": "recall",
    "class1_f1": "f1"
}

baseline_cv = cross_validate(
    baseline_xgb,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("CV Accuracy:", round(baseline_cv["test_accuracy"].mean(), 4))
print("CV Macro F1:", round(baseline_cv["test_macro_f1"].mean(), 4))
print("CV Class 1 Recall:", round(baseline_cv["test_class1_recall"].mean(), 4))
print("CV Class 1 F1:", round(baseline_cv["test_class1_f1"].mean(), 4))

CV Accuracy: 0.8878
CV Macro F1: 0.6964
CV Class 1 Recall: 0.3332
CV Class 1 F1: 0.4553


## 3. Handle Class Imbalance

Because the positive class is much smaller than the negative class, we test class weighting using `scale_pos_weight`.

The weight is calculated from the training data as:

`number of negative samples / number of positive samples`

In [4]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("scale_pos_weight:", round(scale_pos_weight, 2))

scale_pos_weight: 6.1


In [5]:
weighted_xgb = Pipeline(
    steps=[
        ("preprocessor", build_tree_preprocessor()),
        ("model", XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            eval_metric="logloss"
        ))
    ]
)

weighted_cv = cross_validate(
    weighted_xgb,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("CV Accuracy:", round(weighted_cv["test_accuracy"].mean(), 4))
print("CV Macro F1:", round(weighted_cv["test_macro_f1"].mean(), 4))
print("CV Class 1 Recall:", round(weighted_cv["test_class1_recall"].mean(), 4))
print("CV Class 1 F1:", round(weighted_cv["test_class1_f1"].mean(), 4))

CV Accuracy: 0.7864
CV Macro F1: 0.6827
CV Class 1 Recall: 0.7619
CV Class 1 F1: 0.5012


### Class Weighting Interpretation

Class weighting substantially improves Class 1 Recall, increasing it from 0.3332 to 0.7619.

However, this comes with lower Accuracy and a lower Macro F1 score.

Because Macro F1 is the primary evaluation metric, the fully weighted model does not outperform the baseline XGBoost.

The result suggests that a less aggressive class weight may provide a better balance.


In [8]:
weights_to_test = [1, 2, 3, 4, scale_pos_weight]

weight_results = []

for weight in weights_to_test:
    model = Pipeline(
        steps=[
            ("preprocessor", build_tree_preprocessor()),
            ("model", XGBClassifier(
                n_estimators=200,
                max_depth=4,
                learning_rate=0.05,
                scale_pos_weight=weight,
                random_state=42,
                eval_metric="logloss"
            ))
        ]
    )

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    weight_results.append({
        "scale_pos_weight": round(weight, 2),
        "CV Macro F1": scores["test_macro_f1"].mean(),
        "CV Class 1 Recall": scores["test_class1_recall"].mean(),
        "CV Class 1 F1": scores["test_class1_f1"].mean()
    })

weight_results = pd.DataFrame(weight_results)

weight_results.round(4)

,scale_pos_weight,CV Macro F1,CV Class 1 Recall,CV Class 1 F1
0,1.0,0.6964,0.3332,0.4553
1,2.0,0.7309,0.4845,0.5309
2,3.0,0.7328,0.5766,0.5451
3,4.0,0.7239,0.6634,0.5428
4,6.1,0.6827,0.7619,0.5012


### Class Weight Search Interpretation

Moderate class weighting improves both balanced performance and minority-class detection.

- `scale_pos_weight = 3` achieves the highest CV Macro F1 score: 0.7328.
- Class 1 Recall improves from 0.3332 to 0.5766.
- Class 1 F1 also improves to 0.5451.
- Larger weights increase recall further but reduce overall balanced performance.

Based on Macro F1, `scale_pos_weight = 3` is the best class-weighting setting tested.

## 4. Threshold Adjustment

The default classification threshold is 0.50.

We test different probability thresholds to determine whether a different cutoff can improve Macro F1 while maintaining reasonable Class 1 Recall.

In [9]:
best_weighted_xgb = Pipeline(
    steps=[
        ("preprocessor", build_tree_preprocessor()),
        ("model", XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            scale_pos_weight=3,
            random_state=42,
            eval_metric="logloss"
        ))
    ]
)

best_weighted_xgb.fit(X_train, y_train)

positive_probabilities = best_weighted_xgb.predict_proba(X_valid)[:, 1]

In [11]:
import numpy as np

thresholds = np.arange(0.30, 0.71, 0.05)

threshold_results = []

for threshold in thresholds:
    threshold_pred = (
        positive_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Macro F1": f1_score(
            y_valid,
            threshold_pred,
            average="macro"
        ),
        "Class 1 Recall": recall_score(
            y_valid,
            threshold_pred
        ),
        "Class 1 F1": f1_score(
            y_valid,
            threshold_pred
        )
    })

threshold_results = pd.DataFrame(threshold_results)

threshold_results.round(4)

,Threshold,Macro F1,Class 1 Recall,Class 1 F1
0,0.30,0.6711,0.8187,0.4952
1,0.35,0.7011,0.7719,0.5257
2,0.40,0.7241,0.7266,0.5507
3,0.45,0.7328,0.6631,0.5553
4,0.50,0.7415,0.6148,0.5625
5,0.55,0.7474,0.5755,0.5670
6,0.60,0.7430,0.5181,0.5537
7,0.65,0.7356,0.4668,0.5360
8,0.70,0.7246,0.4094,0.5113


### Threshold Adjustment Interpretation

- Lower thresholds substantially increase Class 1 Recall but reduce overall balanced performance.
- Higher thresholds reduce Class 1 Recall.
- A threshold of `0.55` achieves the highest Macro F1 score: `0.7474`.
- This provides the best balance among the tested thresholds.

Based on Macro F1, the best threshold tested is `0.55`.

## 5. Hyperparameter Tuning

We tune the XGBoost model using `RandomizedSearchCV` with stratified cross-validation.

The goal is to find a better combination of hyperparameters while optimizing Macro F1.

The tuned model keeps `scale_pos_weight = 3`, which performed best in the class-weighting experiments.

In [14]:
from sklearn.model_selection import RandomizedSearchCV

In [15]:
param_distributions = {
    "model__n_estimators": [150, 200, 300, 400],
    "model__max_depth": [3, 4, 5, 6],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

In [16]:
tuning_model = Pipeline(
    steps=[
        ("preprocessor", build_tree_preprocessor()),
        ("model", XGBClassifier(
            scale_pos_weight=3,
            random_state=42,
            eval_metric="logloss"
        ))
    ]
)

random_search = RandomizedSearchCV(
    estimator=tuning_model,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="f1_macro",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               'passthrough',
                                                                               ['year',
                                                                                'household_size',
                                                                                'age_of_respondent']),
                                                                              ('cat',
                                                                               OneHotEncoder(handle_unknown='ignore'),
                                                                               ['country',
                                                                                'location_type',
                                                                                'cellphone_access',
                                                                                'gender_of_respondent',
                                                                                'relationship_...
                                                            multi_strategy=None,
                                                            n_estimators=None,
                                                            n_jobs=None,
                                                            num_parallel_tree=None,
                                                            random_state=42, ...))]),
                   n_iter=12, n_jobs=-1,
                   param_distributions={'model__colsample_bytree': [0.8, 1.0],
                                        'model__learning_rate': [0.03, 0.05,
                                                                 0.1],
                                        'model__max_depth': [3, 4, 5, 6],
                                        'model__n_estimators': [150, 200, 300,
                                                                400],
                                        'model__subsample': [0.8, 1.0]},
                   random_state=42, scoring='f1_macro', verbose=1)

In [17]:
print("Best CV Macro F1:", round(random_search.best_score_, 4))
print("Best Parameters:")
print(random_search.best_params_)

Best CV Macro F1: 0.7354
Best Parameters:
{'model__subsample': 0.8, 'model__n_estimators': 150, 'model__max_depth': 4, 'model__learning_rate': 0.05, 'model__colsample_bytree': 1.0}


## 6. Threshold Adjustment After Tuning

Because hyperparameter tuning changes the model and its predicted probabilities, we re-evaluate the classification threshold using the tuned XGBoost model.

In [18]:
best_tuned_xgb = random_search.best_estimator_

best_tuned_xgb.fit(X_train, y_train)

tuned_probabilities = best_tuned_xgb.predict_proba(X_valid)[:, 1]

In [19]:
tuned_threshold_results = []

for threshold in thresholds:
    tuned_pred = (
        tuned_probabilities >= threshold
    ).astype(int)

    tuned_threshold_results.append({
        "Threshold": round(threshold, 2),
        "Macro F1": f1_score(
            y_valid,
            tuned_pred,
            average="macro"
        ),
        "Class 1 Recall": recall_score(
            y_valid,
            tuned_pred
        ),
        "Class 1 F1": f1_score(
            y_valid,
            tuned_pred
        )
    })

tuned_threshold_results = pd.DataFrame(tuned_threshold_results)

tuned_threshold_results.round(4)

,Threshold,Macro F1,Class 1 Recall,Class 1 F1
0,0.30,0.6729,0.8293,0.4989
1,0.35,0.7004,0.7659,0.5240
2,0.40,0.7225,0.7221,0.5479
3,0.45,0.7328,0.6616,0.5551
4,0.50,0.7401,0.6103,0.5599
5,0.55,0.7468,0.5725,0.5657
6,0.60,0.7438,0.5166,0.5547
7,0.65,0.7334,0.4562,0.5312
8,0.70,0.7222,0.4003,0.5062


### Threshold Adjustment After Tuning

The best threshold remains `0.55`.

- Macro F1: `0.7468`
- Class 1 Recall: `0.5725`
- Class 1 F1: `0.5657`

The tuned model performs very similarly to the previous weighted model on the validation set.

Hyperparameter tuning provides a small improvement in cross-validation performance, while the optimal decision threshold remains unchanged.

## 7. Final Model Comparison

We compare the main stages of XGBoost improvement using Macro F1 as the primary metric.

The comparison includes:

- original XGBoost;
- class-weighted XGBoost;
- tuned weighted XGBoost;
- tuned weighted XGBoost with threshold adjustment.

In [20]:
final_results = pd.DataFrame({
    "Model Version": [
        "Original XGBoost",
        "Weighted XGBoost",
        "Tuned Weighted XGBoost",
        "Tuned + Threshold 0.55"
    ],
    "Macro F1": [
        0.6964,
        0.7328,
        0.7354,
        0.7468
    ],
    "Class 1 Recall": [
        0.3332,
        0.5766,
        None,
        0.5725
    ],
    "Class 1 F1": [
        0.4553,
        0.5451,
        None,
        0.5657
    ]
})

final_results

,Model Version,Macro F1,Class 1 Recall,Class 1 F1
0,Original XGBoost,0.6964,0.3332,0.4553
1,Weighted XGBoost,0.7328,0.5766,0.5451
2,Tuned Weighted XGBoost,0.7354,NaN,NaN
3,Tuned + Threshold 0.55,0.7468,0.5725,0.5657


In [21]:
tuned_cv = cross_validate(
    random_search.best_estimator_,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

tuned_cv_recall = tuned_cv["test_class1_recall"].mean()
tuned_cv_f1 = tuned_cv["test_class1_f1"].mean()

print("Tuned CV Class 1 Recall:", round(tuned_cv_recall, 4))
print("Tuned CV Class 1 F1:", round(tuned_cv_f1, 4))

Tuned CV Class 1 Recall: 0.577
Tuned CV Class 1 F1: 0.5491


In [23]:
final_results = pd.DataFrame({
    "Model Version": [
        "Original XGBoost",
        "Weighted XGBoost",
        "Tuned Weighted XGBoost",
        "Tuned + Threshold 0.55"
    ],
    "Evaluation": [
        "5-Fold CV",
        "5-Fold CV",
        "5-Fold CV",
        "Holdout Validation"
    ],
    "Macro F1": [
        0.6964,
        0.7328,
        0.7354,
        0.7468
    ],
    "Class 1 Recall": [
        0.3332,
        0.5766,
        0.5770,
        0.5725
    ],
    "Class 1 F1": [
        0.4553,
        0.5451,
        0.5491,
        0.5657
    ]
})

final_results

,Model Version,Evaluation,Macro F1,Class 1 Recall,Class 1 F1
0,Original XGBoost,5-Fold CV,0.6964,0.3332,0.4553
1,Weighted XGBoost,5-Fold CV,0.7328,0.5766,0.5451
2,Tuned Weighted XGBoost,5-Fold CV,0.7354,0.5770,0.5491
3,Tuned + Threshold 0.55,Holdout Validation,0.7468,0.5725,0.5657


## 8. Final Model Evaluation

The final candidate is the tuned weighted XGBoost model with a decision threshold of `0.55`.

We evaluate this final configuration on the holdout validation set using the main classification metrics and the confusion matrix.

In [24]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    confusion_matrix,
    classification_report
)

final_pred = (tuned_probabilities >= 0.55).astype(int)

final_accuracy = accuracy_score(y_valid, final_pred)
final_macro_f1 = f1_score(y_valid, final_pred, average="macro")
final_recall = recall_score(y_valid, final_pred)
final_class1_f1 = f1_score(y_valid, final_pred)

print("Final Accuracy:", round(final_accuracy, 4))
print("Final Macro F1:", round(final_macro_f1, 4))
print("Final Class 1 Recall:", round(final_recall, 4))
print("Final Class 1 F1:", round(final_class1_f1, 4))

print("\nClassification Report:")
print(classification_report(y_valid, final_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_valid, final_pred))

Final Accuracy: 0.8763
Final Macro F1: 0.7468
Final Class 1 Recall: 0.5725
Final Class 1 F1: 0.5657

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.93      0.93      4043
           1       0.56      0.57      0.57       662

    accuracy                           0.88      4705
   macro avg       0.74      0.75      0.75      4705
weighted avg       0.88      0.88      0.88      4705


Confusion Matrix:
[[3744  299]
 [ 283  379]]


### Final Model Interpretation

The final tuned and weighted XGBoost model with a decision threshold of `0.55` improves balanced classification performance compared with the original XGBoost model.

- Final Macro F1: `0.7468`
- Class 1 Recall: `0.5725`
- Class 1 F1: `0.5657`
- Accuracy: `0.8763`

The final model identifies substantially more positive cases than the original XGBoost model, while maintaining strong performance on the majority class.

The confusion matrix shows:

- 3744 True Negatives
- 299 False Positives
- 283 False Negatives
- 379 True Positives